# 🧬 Prodigal Gene Prediction Examples

This notebook demonstrates prokaryotic gene prediction using **Prodigal**.

- **Basic prediction** -- Predict genes in a prokaryotic DNA sequence using meta mode
- **Batch processing** -- Analyze multiple sequences in parallel
- **Result inspection** -- Examine ORF fields and Prodigal-specific metrics (RBS, start type, GC content)
- **DataFrame exploration** -- Query and filter results with pandas
- **Exporting** -- Save results as GFF, CSV, or FASTA

## 📥 Imports

## 📦 Shared Data Types

### `ORF`
A single predicted open reading frame.

| Field | Type | Description |
|-------|------|-------------|
| `parent_id` | `str` | Identifier of the parent/input sequence |
| `orf_id` | `str` | Unique ORF identifier within the parent sequence |
| `strand` | `Literal["+", "-"]` | Strand direction |
| `frame` | `int` | Reading frame (`1`, `2`, or `3`) |
| `amino_acid_sequence` | `str` | Translated protein sequence |
| `nucleotide_sequence` | `str` | DNA sequence of the ORF |
| `amino_acid_length` | `int` | Length of protein in amino acids |
| `nucleotide_length` | `int` | Length of ORF in nucleotides |
| `nucleotide_start` | `int` | Start position (1-indexed, inclusive) |
| `nucleotide_end` | `int` | End position (1-indexed, inclusive) |
| `metrics` | `Dict[str, Any]` | Tool-specific metrics (includes `start_type`, `rbs_motif`, `rbs_spacer`, `partial_begin`, `partial_end`, `description`) |
| `id` | `str` | Combined identifier: `parent_id` + `orf_id` (computed) |
| `gc_content` | `float` | GC content percentage (computed) |

In [1]:
from bio_programming_tools import ProdigalConfig, ProdigalInput, run_prodigal_prediction


## 🔬 `run_prodigal_prediction`

Predict genes in prokaryotic DNA sequences using Prodigal.

### API Reference

**Input: `ProdigalInput`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `input_sequences` | `List[str]` | *required* | DNA sequence(s) to analyze for open reading frames |

**Config: `ProdigalConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `meta_mode` | `bool` | `True` | Use meta mode for short sequences/fragments (`True`) or single-genome mode (`False`) |
| `translation_table` | `int` | `11` | NCBI translation table (1-25; default `11` for bacteria) |
| `closed_ends` | `bool` | `False` | Prevent genes from running off sequence edges (`True` for complete circular genomes) |

**Output: `ProdigalOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `predicted_orfs` | `List[List[ORF]]` | List of ORF results per input sequence |
| `num_orfs` | `int` | Total number of ORFs predicted (computed) |
| `num_orfs_per_sequence` | `List[int]` | Number of ORFs per input sequence (computed) |
| `results_df` | `pd.DataFrame` | All ORF results as a pandas DataFrame (computed) |

---

### 1. Basic Gene Prediction

Predict genes in a prokaryotic DNA sequence using meta mode (best for short sequences).

In [2]:
# A realistic bacterial gene fragment
sequence = (
    "ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG"
    "GAAAACCCTGGCGTTACCCAACTTAATCGCCTTGCAGCACATCCCCCTTTC"
    "GCCAGCTGGCGTAATAGCGAAGAGGCCCGCACCGATCGCCCTTCCCAACAG"
    "TTGCGCAGCCTGAATGGCGAATGGCGCTTTGCCTGGTTTCCGGCACCAGAA"
    "GCGGTGCCGGAAAGCTGGCTGGAGTGCGATCTTCCTGAGGCCGATACTGTC"
    "GTCGTCCCCTCAAACTGGCAGATGCACGGTTACGATGCGCCCATCTACACC"
    "AACGTGACCTATCCCATTACGGTCAATCCGCCGTTTGTTCCCACGGAGAAT"
    "CCGACGGGTTGTTACTCGCTCACATTTAATGTTGATGAAAGCTGGCTACAG"
    "GAAGGCCAGACGCGAATTATTTTTGATGGCGTTAACTCGGCGTTTCATCTG"
    "TGGTGCAACGGGCGCTGGGTCGGTTACGGCCAGGACAGTCGTTTGCCGTCT"
    "TAA"
)

inputs = ProdigalInput(input_sequences=sequence)
config = ProdigalConfig(meta_mode=True)

result = run_prodigal_prediction(inputs, config)
print(f"Found {result.num_orfs} genes")
result.results_df

Found 1 genes


,parent_id,orf_id,strand,frame,amino_acid_sequence,nucleotide_sequence,amino_acid_length,nucleotide_length,nucleotide_start,nucleotide_end,metrics,id,gc_content,start_type,rbs_motif,rbs_spacer,partial_begin,partial_end,description,partial
0,seq_0,gene_1,+,1,MTMITDSLAVVLQRRDWENPGVTQLNRLAAHPPFASWRNSEEARTD...,ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTG...,170,513,1,513,"{'start_type': 'Edge', 'rbs_motif': None, 'rbs...",seq_0_gene_1,55.360624,Edge,None,None,1,0,1 # 1 # 513 # 1 # ID=1;partial=01_00;start_typ...,01_00


### 2. Batch Processing with Multiple Sequences

In [3]:
sequences = [
    "ATGCGTAAATAA" * 50,
    "ATGGCATAA" * 50,
    "ATGAAACGT" * 50,
]

inputs_batch = ProdigalInput(input_sequences=sequences)
config_batch = ProdigalConfig(meta_mode=True, num_threads=2)

result_batch = run_prodigal_prediction(inputs_batch, config_batch)
print(f"Total genes: {result_batch.num_orfs}")
print(f"Genes per sequence: {result_batch.num_orfs_per_sequence}")

Total genes: 3
Genes per sequence: [1, 1, 1]


### 3. Examining ORF Metrics

Prodigal provides rich annotations including RBS motifs, start types, and GC content.

In [4]:
if result.num_orfs > 0:
    orf = result.predicted_orfs[0][0]
    print(f"Parent ID: {orf.parent_id}")
    print(f"Gene ID: {orf.orf_id}")
    print(f"Strand: {orf.strand}")
    print(f"Frame: {orf.frame}")
    print(f"Position: {orf.nucleotide_start}-{orf.nucleotide_end}")
    print(f"GC content: {orf.gc_content:.3f}")
    print(f"Start type: {orf.start_type}")
    print(f"RBS motif: {orf.rbs_motif}")
    print(f"RBS spacer: {orf.rbs_spacer}")
    print(f"Partial (begin/end): {orf.partial_begin}/{orf.partial_end}")
    print(f"Protein: {orf.amino_acid_sequence[:50]}...")

Parent ID: seq_0
Gene ID: gene_1
Strand: +
Frame: 1
Position: 1-513
GC content: 55.361
Start type: Edge
RBS motif: None
RBS spacer: None
Partial (begin/end): 1/0
Protein: MTMITDSLAVVLQRRDWENPGVTQLNRLAAHPPFASWRNSEEARTDRPSQ...


### 4. DataFrame Exploration

In [5]:
df = result_batch.results_df
if not df.empty:
    print(f"Columns: {list(df.columns)}")
    print(f"\nGenes by strand:")
    print(df['strand'].value_counts())
    print(f"\nAverage GC content: {df['gc_content'].mean():.3f}")
    df.head()

Columns: ['parent_id', 'orf_id', 'strand', 'frame', 'amino_acid_sequence', 'nucleotide_sequence', 'amino_acid_length', 'nucleotide_length', 'nucleotide_start', 'nucleotide_end', 'metrics', 'id', 'gc_content', 'start_type', 'rbs_motif', 'rbs_spacer', 'partial_begin', 'partial_end', 'description', 'partial']

Genes by strand:
strand
+    2
-    1
Name: count, dtype: int64

Average GC content: 30.672


## 💾 Export Results

Save results to files for downstream analysis.

- **GFF** -- Standard gene annotation format
- **CSV** -- Tabular format with all ORF fields and metrics
- **FAA** -- Protein sequences in FASTA format

In [6]:
# Export as GFF (default format for gene annotations)
result.export("prodigal_results", export_path="./example_output", file_format="gff")
print("Exported to ./example_output/prodigal_results.gff")

# Export as CSV
result.export("prodigal_results", export_path="./example_output", file_format="csv")
print("Exported to ./example_output/prodigal_results.csv")

# Export protein sequences as FASTA
result.export("prodigal_results", export_path="./example_output", file_format="faa")
print("Exported to ./example_output/prodigal_results.faa")

Exported to ./example_output/prodigal_results.gff
Exported to ./example_output/prodigal_results.csv
Exported to ./example_output/prodigal_results.faa
